### Ultralytics 실습
#### YOLO모델을 활용한 학습

In [2]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")     # pytorch nano

# model.info(detailed=True)
model.info()

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8550912)

### 모델 학습
- ultralytics YOLO 모델

In [3]:
# datasets 폴더여야 함. (이미지 50개 정도여야함.)
model.train(data="datasets/data.yaml", epochs=100)

New https://pypi.org/project/ultralytics/8.4.138 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.123  Python-3.12.14 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001C4BC0A0F50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,   

### 모델 객체 탐지

In [6]:
# model 로딩
from ultralytics import YOLO
# 학습한 모델의 위치
model_path = YOLO("runs/detect/train/weights/best.pt")
model = YOLO(model_path)
model.info()

Model summary: 130 layers, 3,012,798 parameters, 0 gradients, 8.2 GFLOPs


(130, 3012798, 0, 8.2013696)

In [7]:
import cv2
image_path = './images/number2.jpg'
# image_path = './images/number1.jpg'
img = cv2.imread(image_path)
result = model.predict(image_path)
for box in result[0].boxes:
    # CPU로 학습했을때
    # x1, y1, x2, y2 = box.xyxy[0].numpy().flatten().astype(int)
    # GPU → CPU 이동 후 numpy 변환
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
    cv2.rectangle(img, (x1,y1), (x2,y2), (0, 0, 255), 2)
    cv2.putText(img, f'{result[0].names[int(box.cls)]} {float(box.conf):.4f}', (x1,y1), cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 0, 255))
cv2.imshow('image', img)
cv2.waitKey()
cv2.destroyAllWindows()


image 1/1 C:\yolo_ex\images\number2.jpg: 384x640 1 1, 4 2s, 1 3, 2 4s, 4 6s, 2 7s, 1 8, 15.7ms
Speed: 3.5ms preprocess, 15.7ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)


### 실시간 객체 탐지1 (CPU)

In [10]:
import cv2
import time
import numpy as np

# (선택) 모델이 아직 로드 안 되어 있다면 주석 해제해서 사용하세요.
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")  # 혹은 당신의 커스텀 가중치 경로

from ultralytics import YOLO
# 학습한 모델의 위치
model = YOLO("runs/detect/train/weights/best.pt")

# ===== 설정 =====
cam_index = 0                 # 웹캠 인덱스 (외장 카메라면 1, 2…)
conf_thresh = 0.25            # 표시할 최소 신뢰도
show_fps = True               # 좌상단 FPS 표시
draw_thickness = 2            # 박스 두께
font_scale = 1.2              # 라벨 글자 크기

# ===== 캡처 초기화 =====
cap = cv2.VideoCapture(cam_index)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다 (index={cam_index}).")

# (선택) 해상도 지정
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

prev_time = time.time()

try:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            print("프레임을 가져오지 못했습니다.")
            break

        # YOLO 추론 (결과는 리스트; 첫 결과만 사용)
        results = model.predict(frame, verbose=False)
        res = results[0]

        # 클래스 이름 맵
        names = res.names if hasattr(res, "names") else getattr(model, "names", None)

        # 바운딩 박스 그리기
        if hasattr(res, "boxes") and res.boxes is not None:
            for box in res.boxes:
                conf = float(box.conf)
                if conf < conf_thresh:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                cls_id = int(box.cls)
                label = names[cls_id] if names and cls_id in names else str(cls_id)
                txt = f"{label} {conf:.2f}"

                # 박스 & 라벨
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), draw_thickness)
                cv2.putText(frame, txt, (x1, max(y1-5, 0)),
                            cv2.FONT_HERSHEY_PLAIN, font_scale, (0, 0, 255), 2)

        # FPS 표시
        if show_fps:
            now = time.time()
            fps = 1.0 / (now - prev_time) if now != prev_time else 0.0
            prev_time = now
            cv2.putText(frame, f"FPS: {fps:.1f}", (10, 25),
                        cv2.FONT_HERSHEY_PLAIN, 1.4, (255, 255, 255), 2)

        cv2.imshow("Real-time Detection", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord('q'):  # ESC 또는 q로 종료
            break

finally:
    cap.release()
    cv2.destroyAllWindows() 

### 실시간 객체 탐지2 (GPU)

In [9]:
import cv2
print("OpenCV version:", cv2.__version__)
model = YOLO("runs/detect/train/weights/best.pt")
 
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # 0: 기본 카메라, 1: 외장 카메라
 
while cap.isOpened():
    ret, frame = cap.read() # True/False, ndarray/None
    if (not ret) or (cv2.waitKey(1)==27): # 27키코드는 ESC키
        print("프레임이 없거나 ESC키가 눌려짐")
        break
 
    # 영상처리 코드 삽입
    result = model.predict(frame, verbose=False, conf=0.25)
    for box in result[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().flatten().astype(int)
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0, 0, 255), 2)
        cv2.putText(frame, f'{result[0].names[int(box.cls)]} {float(box.conf):.4f}', 
            (x1,y1), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255))
 
    cv2.imshow("Camera", frame)
 
cap.release()
cv2.destroyAllWindows()

OpenCV version: 4.12.0
프레임이 없거나 ESC키가 눌려짐
